# Build Vector Store — Dukcapil RAG

Notebook ini dijalankan **sekali** untuk:
1. Load `cleaned_docs.pkl` hasil preprocessing
2. Chunking Q&A-aware untuk BAB II + character splitter untuk BAB I/III
3. Embed semua chunks pakai Gemini `text-embedding-004`
4. Persist ke ChromaDB di `data/dukcapil_vector_store/`

Output: vectorstore yang siap dipakai `rag_chat.ipynb` tanpa perlu re-embed.

## Step 1 — Load cleaned docs

In [1]:
import pickle
from pathlib import Path
from collections import Counter

with open("../data/cleaned_docs.pkl", "rb") as f:
    cleaned_docs = pickle.load(f)

print(f"Loaded {len(cleaned_docs)} cleaned docs")
print("Distribusi section:")
for section, count in Counter(d.metadata['section'] for d in cleaned_docs).items():
    print(f"  {section:40s} : {count:3d} halaman")

Loaded 258 cleaned docs
Distribusi section:
  Kata Pengantar                           :   2 halaman
  BAB I - Pendahuluan                      :   7 halaman
  BAB II - Pertanyaan dan Jawaban          : 246 halaman
  BAB III - Penutup                        :   3 halaman


## Step 2 — Q&A-Aware Chunking

**Insight kunci**: jawaban di BAB II melintasi batas halaman. Kita gabungkan semua halaman per-section dulu, baru split.

- **BAB II**: split berdasarkan pola pertanyaan `<no>. <Apa/Bagaimana/...>` → tiap chunk = 1 Q+A utuh
- **BAB I, III, Kata Pengantar**: `RecursiveCharacterTextSplitter` biasa

In [2]:
def concat_section(docs, section_name):
    """
    Gabung semua page_content dari section yang sama jadi satu string panjang.
    Return: (full_text, offset_to_page_map)
    offset_to_page_map: list of (char_offset, page_number) untuk lookup halaman by char index.
    """
    section_docs = sorted(
        [d for d in docs if d.metadata['section'] == section_name],
        key=lambda d: d.metadata['page']
    )
    full_text = ""
    offset_map = []
    for doc in section_docs:
        offset_map.append((len(full_text), doc.metadata['page']))
        full_text += doc.page_content + "\n\n"
    return full_text, offset_map


def page_at_offset(offset_map, char_idx):
    """Cari nomor halaman untuk char index tertentu (binary-search-ish)."""
    page = offset_map[0][1]
    for off, p in offset_map:
        if off > char_idx:
            break
        page = p
    return page


bab2_text, bab2_offsets = concat_section(cleaned_docs, "BAB II - Pertanyaan dan Jawaban")
print(f"BAB II concat length: {len(bab2_text):,} chars")
print(f"First 500 chars:\n{bab2_text[:500]}")

BAB II concat length: 228,065 chars
First 500 chars:
BAB II PERTANYAAN DAN JAWABAN A. IDENTITAS PENDUDUK
1 Bagaimana pencatatan biodata penduduk dalam wilayah NKRI?
Jawaban:
Berdasarkan ketentuan Pasal 4 Perpres Nomor 96 Tahun 2018, pencatatan biodata penduduk dapat dilakukan dengan memenuhi persyaratan sebagai berikut:
a. Surat pengantar (asli) dari rukun tetangga dan rukun warga atau yang disebut dengan nama lain; b. Fotokopi dokumen atau bukti peristiwa kependudukan dan peristiwa penting; c. Fotokopi bukti pendidikan terakhir; d. Apabila tidak 


In [3]:
import re
from langchain_core.documents import Document

# Pola pertanyaan: di awal baris, ada nomor (1-3 digit) + kata tanya
QUESTION_PATTERN = re.compile(
    r'(?:^|\n)\s*(\d{1,3})[\.\s]+(Apakah|Apa|Bagaimana|Berapa|Mengapa|Kapan|Di mana|Dimana|Siapa)\b',
    re.IGNORECASE
)

# Pola sub-section heading: "A. IDENTITAS PENDUDUK" dst (huruf besar, kapital)
SUBSECTION_PATTERN = re.compile(
    r'\n\s*([A-Z])\.\s+([A-Z][A-Z\s/&,-]{3,})\s*\n'
)

def find_subsections(text):
    """Return list of (char_offset, label) untuk tiap sub-section heading."""
    return [(m.start(), f"{m.group(1)}. {m.group(2).strip()}") for m in SUBSECTION_PATTERN.finditer(text)]


def subsection_at_offset(subsections, char_idx):
    """Sub-section paling baru sebelum char_idx."""
    label = "Unknown"
    for off, lbl in subsections:
        if off > char_idx:
            break
        label = lbl
    return label


def qa_chunk_bab2(text, offset_map):
    matches = list(QUESTION_PATTERN.finditer(text))
    subsections = find_subsections(text)
    chunks = []
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        chunk_text = text[start:end].strip()
        if len(chunk_text) < 50:
            continue
        # Ekstrak question text saja (sampai tanda ?)
        q_match = re.match(r'\s*\d{1,3}[\.\s]+([^?]+\??)', chunk_text)
        question_text = q_match.group(1).strip() if q_match else ""
        chunks.append(Document(
            page_content=chunk_text,
            metadata={
                "section": "BAB II - Pertanyaan dan Jawaban",
                "subsection": subsection_at_offset(subsections, start),
                "question_number": int(m.group(1)),
                "question_text": question_text[:200],
                "page_start": page_at_offset(offset_map, start),
                "page_end": page_at_offset(offset_map, end - 1),
                "chunk_type": "qa",
            }
        ))
    return chunks


qa_chunks = qa_chunk_bab2(bab2_text, bab2_offsets)
print(f"BAB II: {len(qa_chunks)} Q&A chunks")
print(f"\nSample chunk #5:")
print(f"  Metadata: {qa_chunks[5].metadata}")
print(f"  Content (first 400 chars):\n{qa_chunks[5].page_content[:400]}")

BAB II: 139 Q&A chunks

Sample chunk #5:
  Metadata: {'section': 'BAB II - Pertanyaan dan Jawaban', 'subsection': 'Unknown', 'question_number': 3, 'question_text': 'Bagaimana cara pencantuman pada kolom Status Hubungan Dalam Keluarga (SHDK) untuk kepala keluarga dan anak sambung/anak tiri?', 'page_start': 37, 'page_end': 39, 'chunk_type': 'qa'}
  Content (first 400 chars):
3. Bagaimana cara pencantuman pada kolom Status Hubungan Dalam Keluarga (SHDK) untuk kepala keluarga dan anak sambung/anak tiri?
Jawaban:
Bila anak sambung/anak tiri tersebut adalah anak yang dibawa dari perkawinan yang sah orang tuanya, maka

pencantuman dalam KK pada kolom SHDK bagi anak sambung/anak tiri dicantumkan dengan status anak. Walaupun dalam kolom SHDK tercantum status anak, namun pada


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

narrative_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""],
)

def chunk_narrative_section(docs, section_name):
    text, offsets = concat_section(docs, section_name)
    if not text.strip():
        return []
    raw_chunks = narrative_splitter.split_text(text)
    out = []
    cursor = 0
    for ch in raw_chunks:
        start = text.find(ch, cursor)
        if start == -1:
            start = cursor
        end = start + len(ch)
        out.append(Document(
            page_content=ch,
            metadata={
                "section": section_name,
                "page_start": page_at_offset(offsets, start),
                "page_end": page_at_offset(offsets, max(start, end - 1)),
                "chunk_type": "narrative",
            }
        ))
        cursor = end - 200  # overlap-aware cursor
    return out


bab1_chunks = chunk_narrative_section(cleaned_docs, "BAB I - Pendahuluan")
bab3_chunks = chunk_narrative_section(cleaned_docs, "BAB III - Penutup")
kp_chunks = chunk_narrative_section(cleaned_docs, "Kata Pengantar")

print(f"BAB I       : {len(bab1_chunks)} chunks")
print(f"BAB III     : {len(bab3_chunks)} chunks")
print(f"Kata Pengantar: {len(kp_chunks)} chunks")

c:\Users\Nafisha\Documents\RAGTrial\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BAB I       : 6 chunks
BAB III     : 3 chunks
Kata Pengantar: 2 chunks


In [5]:
import statistics

all_chunks = kp_chunks + bab1_chunks + qa_chunks + bab3_chunks

print(f"TOTAL CHUNKS: {len(all_chunks)}\n")

# Stats
lengths = [len(c.page_content) for c in all_chunks]
print(f"Length stats: min={min(lengths)}, max={max(lengths)}, mean={statistics.mean(lengths):.0f}, median={statistics.median(lengths):.0f}")

# Distribusi by section
by_section = Counter(c.metadata['section'] for c in all_chunks)
print("\nChunks per section:")
for sec, n in by_section.items():
    print(f"  {sec:40s} : {n}")

# Q&A coverage
qa_with_qnum = sum(1 for c in all_chunks if c.metadata.get('question_number') is not None)
print(f"\nChunks dengan question_number: {qa_with_qnum} / {len(qa_chunks)} ({100*qa_with_qnum/len(qa_chunks):.1f}% dari BAB II)")

TOTAL CHUNKS: 150

Length stats: min=379, max=6491, mean=1588, median=1323

Chunks per section:
  Kata Pengantar                           : 2
  BAB I - Pendahuluan                      : 6
  BAB II - Pertanyaan dan Jawaban          : 139
  BAB III - Penutup                        : 3

Chunks dengan question_number: 139 / 139 (100.0% dari BAB II)


In [6]:
# Cek 3 sample chunk BAB II — pastikan Q & A utuh
import random
random.seed(42)
samples = random.sample(qa_chunks, 3)
for i, c in enumerate(samples, 1):
    print(f"=== SAMPLE {i} ===")
    print(f"Sub  : {c.metadata['subsection']}")
    print(f"Q#   : {c.metadata['question_number']}")
    print(f"Q    : {c.metadata['question_text']}")
    print(f"Pages: {c.metadata['page_start']}-{c.metadata['page_end']}")
    print(f"Len  : {len(c.page_content)}")
    print(f"Content:\n{c.page_content[:600]}...\n")

=== SAMPLE 1 ===
Sub  : E. PINDAH DATANG
Q#   : 12
Q    : Apakah diperlukan surat keterangan izin pasangan untuk melakukan pindah ke daerah lain jika pindah tidak bersama pasangan?
Pages: 71-72
Len  : 656
Content:
12 Apakah diperlukan surat keterangan izin pasangan untuk melakukan pindah ke daerah lain jika pindah tidak bersama pasangan?
Jawaban:
Proses penerbitan SKPWNI sebagaimana diatur pada Perpres 96 Tahun
2018 dan Permendagri 108 Tahun 
2019 bahwa pengurusan SKPWNI hanya mensyaratkan fotokopi Kartu Keluarga dan tidak tercantum syarat surat keterangan izin pasangan.
Sumber rujukan:
Pasal 25 ayat (3) Peraturan Presiden Nomor
96 Tahun 2018 tentang Persyaratan dan Tata Cara Pendaftaran Penduduk dan Pencatatan Sipil.
Pasal 28 ayat (1 ) Peraturan Menteri Dalam Negeri Nomor 108 Tahun 2019 Persyaratan Dan...

=== SAMPLE 2 ===
Sub  : Unknown
Q#   : 4
Q    : Bagaimana cara melakukan pencatatan jenis pekerjaan PPPK pada Kartu Keluarga, mengingat bahwa saat ini belum ada kolom khusus untuk j

## Step 3 — Embedding pakai Gemini

Pakai `text-embedding-004` (multilingual, 768d). Pastikan `GEMINI_API_KEY` ada di `.env`.

In [7]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")
api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
assert api_key, "GEMINI_API_KEY (atau GOOGLE_API_KEY) tidak ditemukan di .env"
os.environ["GOOGLE_API_KEY"] = api_key  # langchain-google-genai cari GOOGLE_API_KEY
print(f"API key loaded ({len(api_key)} chars, starts with {api_key[:6]}...)")

API key loaded (39 chars, starts with AIzaSy...)


In [8]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2",  # Model terbaru, multimodal
    task_type="retrieval_document",
    output_dimensionality=768,  # Optional: reduce dari 3072d ke 768d (lebih hemat)
)

# Smoke test
sample_vec = embeddings.embed_query("test")
print(f"Embedding dim: {len(sample_vec)}")
print(f"First 5 values: {sample_vec[:5]}")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Embedding dim: 768
First 5 values: [-0.0066574984, 0.003566764, 0.046355993, 0.008882849, 0.0068013095]


## Step 4 — Build & Persist ChromaDB

Wipe path lama, lalu bangun fresh collection. `Chroma.from_documents` akan otomatis embed semua chunks dalam batch — ini step yang paling lama (~beberapa menit untuk 300+ chunks).

In [9]:
import shutil

VS_PATH = Path("../data/dukcapil_vector_store")
if VS_PATH.exists():
    shutil.rmtree(VS_PATH)
    print(f"Wiped {VS_PATH.resolve()}")
VS_PATH.mkdir(parents=True, exist_ok=True)

In [10]:
import subprocess
import os

vs_path = "../data/dukcapil_vector_store"
if os.path.exists(vs_path):
    subprocess.run(["powershell", "-Command", f"Remove-Item -Path '{os.path.abspath(vs_path)}' -Recurse -Force"], check=False)
    print(f"Deleted {vs_path}")


Deleted ../data/dukcapil_vector_store


In [11]:
from langchain_chroma import Chroma
from pathlib import Path
import time

BATCH_SIZE = 40
VS_PATH = Path("../data/dukcapil_vector_store")
VS_PATH.mkdir(parents=True, exist_ok=True)

vectorstore = None
for batch_idx, i in enumerate(range(0, len(all_chunks), BATCH_SIZE)):
    batch = all_chunks[i:i+BATCH_SIZE]
    batch_num = batch_idx + 1
    total_batches = (len(all_chunks) - 1) // BATCH_SIZE + 1
    
    print(f"Embedding batch {batch_num}/{total_batches} ({len(batch)} chunks)...", end=" ", flush=True)
    
    # Retry logic untuk handle rate limit
    max_retries = 5
    retry_count = 0
    wait_time = 70  # Start dengan 70 detik (aman dari 100/menit limit)
    
    while retry_count <= max_retries:
        try:
            if vectorstore is None:
                vectorstore = Chroma.from_documents(
                    documents=batch,
                    embedding=embeddings,
                    collection_name="dukcapil_qa",
                    persist_directory=str(VS_PATH),
                )
            else:
                vectorstore.add_documents(batch)
            
            print("✓")
            break  # Success
            
        except Exception as e:
            error_msg = str(e)
            if "429" in error_msg or "RESOURCE_EXHAUSTED" in error_msg:
                retry_count += 1
                if retry_count > max_retries:
                    print(f"\n✗ Max retries exceeded after {max_retries} attempts")
                    raise
                print(f"\n  Rate limited. Retry {retry_count}/{max_retries} in {wait_time}s...", end="", flush=True)
                time.sleep(wait_time)
                wait_time = int(wait_time * 1.5)  # Exponential backoff
                print(" retrying...")
            else:
                print(f"\n✗ Error: {error_msg}")
                raise

count = vectorstore._collection.count()
print(f"\n✓ Vector store ready: {count} chunks ter-embed & ter-persist")
assert count == len(all_chunks), f"Mismatch: {count} vs {len(all_chunks)}"

Embedding batch 1/4 (40 chunks)... ✓
Embedding batch 2/4 (40 chunks)... ✓
Embedding batch 3/4 (40 chunks)... 
  Rate limited. Retry 1/5 in 70s... retrying...
✓
Embedding batch 4/4 (30 chunks)... ✓

✓ Vector store ready: 150 chunks ter-embed & ter-persist


## Step 5 — Sanity Check Retrieval

Test similarity search beberapa query untuk konfirmasi bahwa retrieval bekerja.

In [13]:
test_queries = [
    "Bagaimana cara mengurus KTP-el yang hilang?",
    "Syarat membuat akta kelahiran",
    "Pencatatan perkawinan beda agama",
]

for q in test_queries:
    print(f"\n>>> Query: {q}")
    results = vectorstore.similarity_search(q, k=3)
    for i, r in enumerate(results, 1):
        meta = r.metadata
        preview = r.page_content[:150].replace("\n", " ")
        q_info = f" Q#{meta.get('question_number')}" if meta.get('question_number') else ""
        print(f"  [{i}] {meta.get('section', '?')[:20]}{q_info} hal {meta.get('page_start')}\n      {preview}...")


>>> Query: Bagaimana cara mengurus KTP-el yang hilang?
  [1] BAB II - Pertanyaan  Q#4 hal 42
      4. Apakah penerbitan KTP-el dapat dilakukan di luar Kabupaten/Kota alamat domisili yang tertera dalam KKnya? Jawaban: Berdasarkan ketentuan Pasal   Pe...
  [2] BAB II - Pertanyaan  Q#1 hal 40
      1. Bagaimana penerbitan KTP-el pertama kali bagi WNI? Jawaban: Berdasarkan Pasal 15 Perpres Nomor 96 Tahun 2018, penerbitan KTP-el bagi penduduk WNI h...
  [3] BAB II - Pertanyaan  Q#6 hal 44
      6. Apakah pas foto pada KTP-el dapat diganti? Jawaban: Berdasarkan ketentuan Pasal   Permendagri Nomor 74 Tahun 2015, bahwa perubahan elemen data pas ...

>>> Query: Syarat membuat akta kelahiran
  [1] BAB II - Pertanyaan  Q#2 hal 82
      2. Bagaimana pencatatan kelahiran anak yang lahir di luar negeri dan belum memiliki akta kelahiran terbitan luar negeri, sedangkan yang bersangkutan s...
  [2] BAB II - Pertanyaan  Q#4 hal 83
      4. Bagaimana membuat akta kelahiran sebagai anak ayah dan ibu deng

## Done

Vector store ter-persist di `data/dukcapil_vector_store/`. Lanjut ke `rag_chat.ipynb` untuk eksperimen retrieval & generation.